# Questão 3
## 1. Motivação
Como a participação de candidatas e de eleitas varia entre partidos, estados e municípios?

## 2. Hipótese


In [2]:
pip install plotly

  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [4]:
df = pd.read_parquet("../data/resultados_candidato.parquet")
df.shape

FileNotFoundError: [Errno 2] No such file or directory: '../data/resultados_candidato.parquet'

In [ ]:
base = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado federal", "deputado distrital"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)

In [ ]:
def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()

cand = pct_fem(base, ["ano", "cargo"])
eleit = pct_fem(base[base["eleito"]], ["ano", "cargo"])

cmp_ = cand.merge(eleit, on=["ano", "cargo"], suffixes=("_cand", "_eleit"))
cmp_["dif_pp"] = cmp_["pct_fem_eleit"] - cmp_["pct_fem_cand"]

In [ ]:
casos = [(2022, "deputado federal"), (2022, "deputado estadual"), (2024, "vereador")]
for ano, cargo in casos:
    s = (base[(base["ano"] == ano) & (base["cargo"] == cargo)]
         .groupby("sigla_partido", observed=True).size().sort_values(ascending=False))
    print(ano, cargo, "-", len(s), "partidos,", (s >= 100).sum(), "com 100 ou mais candidaturas")
    print(s.head(8), "\n")

# tamanho das bancadas por UF (deputado federal, 2022)
print(base[(base["ano"] == 2022) & (base["cargo"] == "deputado federal") & base["eleito"]]
      .groupby("sigla_uf", observed=True).size().sort_values().head(10))

# câmaras municipais em 2024: quantos municípios e qual o tamanho típico
cam = (base[(base["ano"] == 2024) & (base["cargo"] == "vereador") & base["eleito"]]
       .groupby("id_municipio", observed=True).size())
print(len(cam), cam.describe())